In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# Charger le dataset ArSarcasm-v2
df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("✅ Dataset chargé !")
print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")
print(df_train.columns.tolist())

In [ ]:
#Extraire les embeddings en séquence complète.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERT")
model = AutoModel.from_pretrained("UBC-NLP/MARBERT")
model.eval()

print("✅ MARBERT chargé !")

In [ ]:
import re
import pyarabic.araby as ar

def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www.\S+', '', tweet)
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    tweet = re.sub(r'[^ء-ي\s]', '', tweet)
    tweet = ar.normalize_hamza(tweet)
    tweet = ar.normalize_ligature(tweet)
    tweet = ar.strip_tashkeel(tweet)
    return tweet.strip()

MAX_LENGTH = 64  # longueur de séquence choisie pour limiter le temps de calcul

def get_embedding_sequence(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH,
                       padding="max_length")
    with torch.no_grad():
        outputs = model(**inputs)
    sequence = outputs.last_hidden_state.squeeze(0).numpy()  # (MAX_LENGTH, 768)
    return sequence

# Test rapide sur un exemple
df_train["cleaned_tweet"] = df_train["tweet"].apply(clean_tweet)
df_test["cleaned_tweet"] = df_test["tweet"].apply(clean_tweet)

exemple = df_train["cleaned_tweet"].iloc[0]
seq_test = get_embedding_sequence(exemple)
print(f"Forme de la séquence : {seq_test.shape}")  # doit être (64, 768)
print("✅ Fonction d'extraction en séquence prête !")

In [ ]:
#Vérifier la distribution des longueurs de tweets

In [ ]:
# Vérifier la distribution du nombre de tokens dans le dataset
longueurs = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer.tokenize(t))
)
print(f"Longueur moyenne  : {longueurs.mean():.1f} tokens")
print(f"Longueur médiane  : {longueurs.median():.1f} tokens")
print(f"Longueur max      : {longueurs.max()} tokens")
print(f"95e percentile    : {longueurs.quantile(0.95):.1f} tokens")
print(f"99e percentile    : {longueurs.quantile(0.99):.1f} tokens")

In [ ]:
#Extraction en séquence complète (train + test)

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

from tqdm import tqdm
tqdm.pandas()

print("⏳ Extraction séquences train... (peut prendre du temps)")
train_sequences = df_train["cleaned_tweet"].progress_apply(get_embedding_sequence)

print("⏳ Extraction séquences test...")
test_sequences = df_test["cleaned_tweet"].progress_apply(get_embedding_sequence)

# Convertir en arrays numpy (N, 64, 768)
X_train_seq = np.stack(train_sequences.values)
X_test_seq = np.stack(test_sequences.values)

print("✅ Extraction terminée !")
print(f"Train : {X_train_seq.shape}")
print(f"Test  : {X_test_seq.shape}")

# Sauvegarder au format .npy (plus efficace que CSV pour des matrices)
np.save("../outputs/train_embeddings_sequence.npy", X_train_seq)
np.save("../outputs/test_embeddings_sequence.npy", X_test_seq)

# Sauvegarder aussi les labels correspondants
np.save("../outputs/train_labels.npy", df_train["sarcasm"].astype(int).values)
np.save("../outputs/test_labels.npy", df_test["sarcasm"].astype(int).values)

print("✅ Fichiers sauvegardés dans outputs/ !")